# Voice Ubuntu NLP v6 — Train, Test, and Export in Google Colab

This notebook retrains the v6 classifier from the expanded dataset, runs automated tests and a separate evaluation set, then saves a tested project ZIP **directly to Google Drive**. It does not download the final ZIP to your computer.

## One-time setup
Place `Voice_Ubuntu_NLP_v6_SOURCE.zip` in `My Drive/VoiceUbuntuNLP/`. This ZIP is the source package containing the expanded dataset, scripts, tests, and notebooks. Do not rename it unless you also change the filename in the next cell.

**Safety note:** The NLP module returns structured intent data only. It does not execute Ubuntu commands. The security module must validate every action before execution.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, zipfile, os, sys, subprocess, json

drive.mount("/content/drive")
DRIVE_FOLDER = Path("/content/drive/MyDrive/VoiceUbuntuNLP")
INPUT_ZIP = DRIVE_FOLDER / "Voice_Ubuntu_NLP_v6_SOURCE.zip"
OUTPUT_ZIP = DRIVE_FOLDER / "Voice_Ubuntu_NLP_v6_TESTED.zip"
WORK_ROOT = Path("/content/voice_ubuntu_v6_workspace")

if not INPUT_ZIP.is_file():
    raise FileNotFoundError(
        f"Cannot find {INPUT_ZIP}. Put Voice_Ubuntu_NLP_v6_SOURCE.zip in My Drive/VoiceUbuntuNLP."
    )

if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP) as zf:
    root = WORK_ROOT.resolve()
    for member in zf.infolist():
        target = (WORK_ROOT / member.filename).resolve()
        if target != root and root not in target.parents:
            raise ValueError(f"Unsafe path in source ZIP: {member.filename}")
    zf.extractall(WORK_ROOT)

PROJECT_DIR = next(
    p.parent for p in WORK_ROOT.rglob("train_model.py")
    if (p.parent / "voice_ubuntu_parser.py").is_file()
)
print("Project:", PROJECT_DIR)
print("Source ZIP:", INPUT_ZIP)

## 1. Install dependencies and regenerate the expanded dataset

This starts from the untouched v5 baseline file included for reproducibility and rebuilds the v6 dataset. The independent evaluation set is saved separately and never added to training.

In [ ]:
def run_command(args, cwd=PROJECT_DIR):
    result = subprocess.run(args, cwd=str(cwd), text=True, capture_output=True)
    output = (result.stdout or "") + ("\n" + result.stderr if result.stderr else "")
    print(output[-16000:])
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {args}")
    return output

INSTALL_OUTPUT = run_command([
    sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"
])
DATA_OUTPUT = run_command([sys.executable, "prepare_training_data.py"])
run_command([sys.executable, "-m", "pip", "install", "-q", "pytest>=8,<10"])
print("Dataset preparation complete.")

## 2. Train the v6 model

The training script first reports a stratified random holdout score, then fits the release model on the whole training CSV. Because the data contains template-based phrase variants, the random holdout score can be optimistic; review the separate independent evaluation below too.

In [ ]:
TRAIN_OUTPUT = run_command([sys.executable, "train_model.py"])
metadata = json.loads((PROJECT_DIR / "voice_ubuntu_model_metadata.json").read_text(encoding="utf-8"))
print("\nModel version:", metadata["version"])
print("Training examples:", metadata["training_examples"])
print("Intent count:", metadata["intent_count"])
print("Saved model exists:", (PROJECT_DIR / "voice_ubuntu_intent_model.joblib").is_file())

## 3. Run automated tests

This suite checks the runtime, entity extraction, English/Arabic examples, local-media routing, query parsing, URL encoding, and randomized parser robustness.

In [ ]:
PYTEST_OUTPUT = run_command([
    sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--tb=short"
])
print("Automated tests passed.")

## 4. Run the independent evaluation and parser smoke checks

The 40 examples in `independent_eval_v6.csv` are kept separate from training and checked for exact normalized phrase overlap. This small, hand-written set is an additional sanity check, not a comprehensive real-world benchmark.

In [ ]:
EVAL_OUTPUT = run_command([sys.executable, "evaluate_model_v6.py"])
validation = json.loads((PROJECT_DIR / "final_validation_v6.json").read_text(encoding="utf-8"))
print("Independent accuracy:", validation["independent_eval"]["accuracy"])
print("Independent macro F1:", validation["independent_eval"]["macro_f1"])
print("Exact training overlap:", validation["independent_eval"]["exact_normalized_training_overlap"])
print("Parser smoke tests all passed:", validation["parser_smoke_all_passed"])
print("URL round-trip tests all passed:", validation["url_round_trip_all_passed"])

if not validation["independent_eval_all_correct"]:
    raise RuntimeError("Independent evaluation has failures. Review the report before exporting.")
if not validation["parser_smoke_all_passed"] or not validation["url_round_trip_all_passed"]:
    raise RuntimeError("Parser/URL smoke tests failed. Review the report before exporting.")

## 5. Try your own commands

These examples only parse text; they do not launch apps or execute Linux commands.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
from voice_ubuntu_parser import parse_text

examples = [
    "Open Firefox",
    "Locate my project PDF",
    "Create a folder in Documents called Notes",
    "Play Interstellar",
    "Play Interstellar on YouTube",
    "ممكن تفتح يوتيوب",
    "شغل أغنية تملي معاك",
    "ابحث على يوتيوب عن شرح التفاضل",
    "Open Firefox and search YouTube for rock and roll",
]
for text in examples:
    print("\nINPUT:", text)
    print(json.dumps(parse_text(text), ensure_ascii=False, indent=2))

## 6. Export the tested package to Google Drive

This final cell creates the ZIP only after the test and evaluation stages above completed successfully. It excludes Python caches and virtual environments. The ZIP and a separate report are saved to Drive without downloading them to your computer.

In [ ]:
# Write a combined log inside the project before packaging.
combined_report = [
    "VOICE UBUNTU NLP v6 — COLAB TRAINING AND TEST REPORT",
    "",
    "DEPENDENCY INSTALLATION", INSTALL_OUTPUT,
    "DATASET PREPARATION", DATA_OUTPUT,
    "TRAINING / RANDOM HOLDOUT", TRAIN_OUTPUT,
    "AUTOMATED TESTS", PYTEST_OUTPUT,
    "INDEPENDENT EVALUATION AND PARSER SMOKE", EVAL_OUTPUT,
    "",
    "LIMITATIONS:",
    "The independent evaluation is a small manually curated text set.",
    "The random row split may be optimistic due to template-generated utterances.",
    "No live microphone, speech-recognition, Ubuntu GUI, or operating-system execution was tested.",
    "The parser uses mock application data in some tests.",
    "Security validation by the security module is required before any OS action.",
]
(PROJECT_DIR / "COLAB_TRAIN_TEST_REPORT_V6.txt").write_text("\n".join(map(str, combined_report)), encoding="utf-8")

required = [
    "voice_ubuntu_intent_model.joblib", "nlp_runtime.py", "voice_ubuntu_parser.py",
    "training_examples.csv", "independent_eval_v6.csv", "requirements.txt",
    "train_model.py", "prepare_training_data.py", "evaluate_model_v6.py",
    "FINAL_VALIDATION_REPORT_V6.txt", "COLAB_TRAIN_TEST_REPORT_V6.txt",
]
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Required release files missing: {missing}")

excluded_dirs = {"__pycache__", ".pytest_cache", ".venv", "venv", ".git", ".ipynb_checkpoints", ".mypy_cache", ".ruff_cache"}
excluded_suffixes = {".pyc", ".pyo"}
temp_zip = DRIVE_FOLDER / "Voice_Ubuntu_NLP_v6_TESTED_TEMP.zip"
temp_zip.unlink(missing_ok=True)

with zipfile.ZipFile(temp_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PROJECT_DIR.rglob("*")):
        relative = path.relative_to(PROJECT_DIR)
        if any(part in excluded_dirs for part in relative.parts) or not path.is_file() or path.suffix in excluded_suffixes:
            continue
        archive.write(path, arcname=str(Path("Voice_Ubuntu_NLP_v6") / relative))

with zipfile.ZipFile(temp_zip, "r") as archive:
    corrupt = archive.testzip()
    archived = set(archive.namelist())
    required_archive = [f"Voice_Ubuntu_NLP_v6/{name}" for name in required]
    missing_archive = [name for name in required_archive if name not in archived]
    if corrupt or missing_archive:
        temp_zip.unlink(missing_ok=True)
        raise RuntimeError(f"ZIP validation failed. Corrupt={corrupt}; missing={missing_archive}")

temp_zip.replace(OUTPUT_ZIP)
print("\nSUCCESS: tested package generated directly in Google Drive")
print("ZIP:", OUTPUT_ZIP)
print("Size MB:", round(OUTPUT_ZIP.stat().st_size / (1024 * 1024), 2))